**Import Libraries**

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

**Load Dataset**

In [ ]:
from pathlib import Path

DATA = Path("../data/processed")

df = pd.read_csv(DATA/"cleaned_dataset.csv")

print(df.shape)

df.head()

**Data Set Overview**

In [ ]:
print(df.info())
df.describe(include="all").T

**Missing Value Analysis**

In [ ]:
missing = pd.DataFrame({

    "Missing Count":df.isna().sum(),

    "Missing %":(df.isna().mean()*100).round(2)

})

missing.sort_values(
    "Missing %",
    ascending=False
)

**Missing Heatmap**

In [ ]:
plt.figure(figsize=(15,6))

sns.heatmap(
    df.isnull(),
    cbar=False
)

plt.title("Missing Data Pattern")

plt.show()

**Missing Barplot**

In [ ]:
missing = missing.sort_values(
    "Missing %",
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    x=missing["Missing %"],
    y=missing.index
)

plt.title("Percentage Missing")

plt.show()

**Duplicate Analysis**

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicates:",duplicates)

**Target Variable Analysis**

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df["VO2max"],
    bins=30,
    kde=True
)

plt.title("VO₂max Distribution")

plt.show()
sns.kdeplot(
    df["VO2max"],
    fill=True
)
sns.kdeplot(
    df["VO2max"],
    fill=True
)
stats.probplot(
    df["VO2max"].dropna(),
    dist="norm",
    plot=plt
)

plt.show()
stat,p = stats.shapiro(
    df["VO2max"].dropna().sample(
        min(500,len(df.dropna()))
    )
)

print(stat)
print(p)
df["VO2max"].quantile(
    [0.01,
     0.05,
     0.25,
     0.50,
     0.75,
     0.95,
     0.99]
)
df["Fitness_Group"]=pd.cut(

    df["VO2max"],

    bins=[0,30,45,100],

    labels=[
        "Low",
        "Moderate",
        "High"
    ]
)
sns.countplot(
    data=df,
    x="Fitness_Group"
)

**Demographic Analysis**

In [ ]:
sns.histplot(
    df["Age"],
    bins=25
)
sns.countplot(
    data=df,
    x="Sex"
)
sns.scatterplot(

    data=df,

    x="Age",

    y="VO2max"
)
sns.regplot(

    data=df,

    x="Age",

    y="VO2max",

    scatter=False,

    color="red"
)
stats.pearsonr(

    df["Age"],

    df["VO2max"]
)

**Sex Differences**

In [ ]:
sns.boxplot(

    data=df,

    x="Sex",

    y="VO2max"
)
df.groupby("Sex")[
    "VO2max"
].describe()
stats.ttest_ind(

    df[df.Sex=="Male"]["VO2max"],

    df[df.Sex=="Female"]["VO2max"],

    equal_var=False
)

**BMI Analysis**

In [ ]:
sns.histplot(
    df["BMI"]
)
sns.countplot(

    data=df,

    x="BMI_Category"
)
sns.scatterplot(

    data=df,

    x="BMI",

    y="VO2max"
)
sns.regplot(

    data=df,

    x="BMI",

    y="VO2max",

    scatter=False,

    color="red"
)
stats.pearsonr(

    df["BMI"],

    df["VO2max"]
)

**Excercise Test Variables**

In [ ]:
exercise_vars=[

"Warmup_Time",

"Stage1_Time",

"Stage2_Time",

"Max_HR"
]

for col in exercise_vars:

    sns.histplot(df[col])

    plt.title(col)

    plt.show()

for col in exercise_vars:

    sns.scatterplot(

        data=df,

        x=col,

        y="VO2max"
    )

    plt.show()

**Correlation Matrix**

In [ ]:
numeric=df.select_dtypes(
    include=np.number
)
corr=numeric.corr()
plt.figure(figsize=(12,10))

sns.heatmap(

corr,

annot=True,

fmt=".2f",

cmap="coolwarm"

)

**Pairplot**

In [ ]:
sns.pairplot(

df[

["Age",

"BMI",

"Weight_kg",

"Height_cm",

"VO2max"]

]
)

**Outlier Detection**

In [ ]:
Q1=df["VO2max"].quantile(.25)

Q3=df["VO2max"].quantile(.75)

IQR=Q3-Q1
lower=Q1-1.5*IQR

upper=Q3+1.5*IQR
outliers=df[
(df["VO2max"]<lower)|
(df["VO2max"]>upper)
]
for col in [

"Age",

"BMI",

"Weight_kg",

"Height_cm",

"VO2max"

]:

    sns.boxplot(

        x=df[col]
    )

    plt.show()

**Statistical Summary**

In [ ]:
summary = pd.DataFrame({
    "Mean": numeric.mean(),
    "Median": numeric.median(),
    "Std": numeric.std(),
    "Min": numeric.min(),
    "Max": numeric.max(),
    "Skewness": numeric.skew(),
    "Kurtosis": numeric.kurt()
})

summary.round(3)
summary.to_csv("../results/EDA_Summary.csv")

At the end of the notebook, summarize:

VO₂max Distribution: Is it approximately normal or skewed?
Age Effect: Correlation coefficient and significance.
Sex Differences: Mean difference and t-test results.
BMI Relationship: Correlation with VO₂max.
Exercise Variables: Which variables appear most associated with VO₂max?
Outliers: Number and whether they appear biologically plausible.
Missing Data: Overall percentage and variables most affected.